In [5]:
import pandas as pd
import numpy as np

SEED = 42
N_FOLDS= 5

df=pd.read_csv("../data/raw/cs-training.csv")
print(df.shape)
df.head()
print("....Column list...")
print(df.columns.tolist())
print("...Column types....")
print(df.dtypes)
print("...Missinf values....")
print(df.isna().sum())

(150000, 11)
....Column list...
['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']
...Column types....
SeriousDlqin2yrs                          int64
RevolvingUtilizationOfUnsecuredLines    float64
age                                       int64
NumberOfTime30-59DaysPastDueNotWorse      int64
DebtRatio                               float64
MonthlyIncome                           float64
NumberOfOpenCreditLinesAndLoans           int64
NumberOfTimes90DaysLate                   int64
NumberRealEstateLoansOrLines              int64
NumberOfTime60-89DaysPastDueNotWorse      int64
NumberOfDependents                      float64
dtype: object
...Missinf values....
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

TARGET="SeriousDlqin2yrs"
X=df.drop(columns=[TARGET])
y=df[TARGET]

skf=StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof=np.zeros(len(df))

for fold, (tr_idx, va_idx) in enumerate (skf.split(X,y)):
    model=lgb.LGBMClassifier(random_states=SEED, verbose=-1)
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    print(fold, roc_auc_score(y.iloc[va_idx], oof[va_idx]))

print("CV AUC:", roc_auc_score(y, oof))

0 0.8632581750355136
1 0.8640757367164336
2 0.8693250531467367
3 0.8586230619201238
4 0.8688919690414969
CV AUC: 0.8647238695530333


In [9]:
def run_cv(X, y, params=None):
    params = params or {}
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))
    for tr_idx, va_idx in skf.split(X, y):
        model = lgb.LGBMClassifier(random_state=SEED, verbose=-1, **params)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    score = roc_auc_score(y, oof)
    print("CV AUC:", round(score, 5))
    return oof, score